In [1]:
import os

os.environ["USE_TF"] = "0"
os.environ["USE_JAX"] = "0"

import json
import torch
import soundfile as sf
import unicodedata
import re
from pathlib import Path
from tqdm import tqdm

from transformers import WhisperProcessor, WhisperForConditionalGeneration
from jiwer import wer, cer
import lightning as L


In [2]:
!pip install transformers datasets jiwer soundfile -q
!pip install lightning>=2.0.0 -q
!pip install unbabel-comet -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.9 which is incompatible.
tensorflow 2.20.0 requires protobuf>=5.28.0, but you have protobuf 4.25.9 which is incompatible.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.9 which is incompatible.


In [11]:
from google.colab import drive
drive.mount('/content/drive')

# CHECKPOINT_PATH = "/content/drive/MyDrive/audio_lab/wer=0.200.ckpt"

PREPARED_DATA_ROOT = Path("/content/drive/MyDrive/prepared_stt")

MODEL_NAME = "openai/whisper-small"

Mounted at /content/drive


In [12]:
def normalize_text(text: str, keep_punctuation: bool = False) -> str:
    text = unicodedata.normalize('NFKC', text)
    text = text.replace('\u0301', '')
    text = text.replace('’', "'").replace('`', "'").replace('ʼ', "'")
    text = text.lower().strip()
    if not keep_punctuation:
        allowed = set(" абвгґдеєжзиіїйклмнопрстуфхцчшщьюя'")
        text = ''.join([ch if ch in allowed else ' ' for ch in text])
    return re.sub(r'\s+', ' ', text).strip()

class WhisperModule(L.LightningModule):
    def __init__(self, model_name: str, processor: WhisperProcessor):
        super().__init__()
        self.processor = processor
        self.model = WhisperForConditionalGeneration.from_pretrained(model_name)

    def forward(self, input_features):
        return self.model.generate(input_features)

In [13]:
print("Завантаження моделі...")
# model_module = WhisperModule.load_from_checkpoint(
#     CHECKPOINT_PATH,
#     model_name=MODEL_NAME,
#     processor=processor
# )
# model_module.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
# model_module.to(device)

print("Завантаження базової (pretrained) моделі...")
processor = WhisperProcessor.from_pretrained(MODEL_NAME)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

model.eval()
model.to(device)

test_lines = [
    'toronto_27', 'toronto_46', 'toronto_42', 'toronto_37', 'toronto_89',
    'toronto_43', 'toronto_157', 'toronto_9', 'toronto_156', 'toronto_7',
    'toronto_123', 'toronto_54', 'toronto_67', 'toronto_62', 'toronto_81',
    'toronto_134', 'toronto_148', 'toronto_21', 'toronto_135', 'toronto_166',
    'toronto_58'
]

test_samples = []
manifest_path = PREPARED_DATA_ROOT / "manifests" / "test.jsonl"

with open(manifest_path, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        if item['speaker_id'] in test_lines:
            file_name = Path(item["audio_filepath"]).name
            correct_path = PREPARED_DATA_ROOT / "audio" / "test" / file_name

            item["audio_filepath"] = str(correct_path)
            test_samples.append(item)

print(f"Знайдено {len(test_samples)} записів. Шлях до першого файлу: {test_samples[0]['audio_filepath']}")

Завантаження моделі...
Завантаження базової (pretrained) моделі...
Знайдено 591 записів. Шлях до першого файлу: /content/drive/MyDrive/prepared_stt/audio/test/dataset__toronto_157__toronto_157_0.wav


In [14]:
import os
test_file = test_samples[0]['audio_filepath']
print(f"Файл існує? {os.path.exists(test_file)}")

Файл існує? True


In [16]:
hyps, refs, sources = [], [], []

for item in tqdm(test_samples, desc="Inference"):
    audio_path = item["audio_filepath"]
    audio, sr = sf.read(audio_path)

    input_features = processor(audio, sampling_rate=sr, return_tensors="pt").input_features.to(device)

    with torch.no_grad():
        generated_ids = model.generate(input_features)
        transcription = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

    hyps.append(normalize_text(transcription))
    refs.append(item["text"])
    sources.append(item.get("original_text", item["text"]))

final_wer = wer(refs, hyps)
final_cer = cer(refs, hyps)

print(f"\nРезультати класичних метрик:")
print(f"WER: {final_wer * 100:.2f}%")
print(f"CER: {final_cer * 100:.2f}%")


Inference:   0%|          | 0/591 [00:00<?, ?it/s]Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.

Inference: 100%|██████████| 591/591 [13:09<00:00,  1.34s/it]


Результати класичних метрик:
WER: 42.37%
CER: 20.85%


In [17]:
from comet import download_model, load_from_checkpoint as load_comet

print("\nЗавантаження та запуск COMET...")
comet_model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_comet(comet_model_path)

comet_data = [
    {"src": s, "mt": h, "ref": r}
    for s, h, r in zip(sources, hyps, refs)
]

comet_output = comet_model.predict(comet_data, batch_size=8, gpus=1 if device=="cuda" else 0)
print(f"COMET System Score: {comet_output.system_score:.4f}")


Завантаження та запуск COMET...


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

checkpoints/model.ckpt:   0%|          | 0.00/2.32G [00:00<?, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages

COMET System Score: 0.7523


In [ ]:
report = {
    "wer": final_wer,
    "cer": final_cer,
    "comet": comet_output.system_score,
    "examples": [{"ref": r, "hyp": h} for r, h in zip(refs[:5], hyps[:5])]
}

with open("final_evaluation_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)